In [ ]:
import os
import pandas as pd
import matplotlib as mp
import numpy as np

### Best Weights

In [ ]:
df = pd.read_csv("best_weights_mrr10.csv")

df.head
len(df)

In [ ]:
queries_data = "/Users/hannahzhang/Desktop/Github Repos/ERSP-TeamYang/data/queries/queries.dev.tsv"

In [ ]:
queries_df = pd.read_csv(queries_data, sep="\t", names=['Query Id', 'Query'])

print(queries_df)


### Word/ Character Count

In [ ]:
queries = df['Query ID']

char_count_list = []
word_count_list = []

for query in queries:
    query = queries_df.loc[queries_df["Query Id"] == query]["Query"].to_string(index=False).strip()

    char_count = 0
    for char in query:
        if char.isalnum():
            char_count += 1

    word_list = query.split(" ")
    word_count = len(word_list)

    char_count_list.append(char_count)
    word_count_list.append(word_count)


df['Character Count'] = char_count_list 
df['Word Count'] = word_count_list

df.head()

### Question Type Keywords (what, when, etc...)

In [ ]:
what_list = [0] * 6980
where_list = [0] * 6980
when_list = [0] * 6980
why_list = [0] * 6980
how_list = [0] * 6980

queries_subset = queries_df[queries_df["Query Id"].isin(queries)]["Query"]

for idx in range(len(queries_subset)):
    word_list = queries_subset.iloc[idx].lower().split(" ")

    if "what" in word_list:
        what_list[idx] = 1

    if "where" in word_list:
        where_list[idx] = 1

    if "when" in word_list:
        when_list[idx] = 1

    if "why" in word_list:
        why_list[idx] = 1

    if "how" in word_list:
        how_list[idx] = 1

df['What'] = what_list
df['Where'] = where_list
df['When'] = when_list
df['Why'] = why_list
df['How'] = how_list

df


### CLS Tokens

In [ ]:
tokens_df = pd.read_csv("../cls_token_split_files/cls_tokens.csv", sep=',', names=[f"Index {i}" for i in range(768)])
tokens_array = tokens_df.values

In [ ]:
tokens_df

In [ ]:
df['CLS Token'] = [np.zeros(100)] * 6980
df

In [ ]:
for i in range(6980):
    df['CLS Token'][i] = tokens_array[i] # cls token array column

In [ ]:
df

In [ ]:
# df['CLS Tokens'] = tokens_array.tolist() # cls token one column

In [ ]:
alt_df = df.join(tokens_df) # cls token multiple columns
alt_df = alt_df.drop(["CLS Token"], axis=1)

In [ ]:
alt_df

In [ ]:
print(alt_df.columns)
alt_df

print("CLS Token" in alt_df.columns)

In [ ]:
questions = queries_df["Query"].to_list()
for question in questions:
    print(question)

### Question Type

In [ ]:
# from transformers import BertTokenizer, BertForSequenceClassification
# import torch

# question_type = []

# # Load model
# model_name = "PrimeQA/tydiqa-boolean-question-classifier"
# tokenizer = BertTokenizer.from_pretrained(model_name)
# model = BertForSequenceClassification.from_pretrained(model_name)

# for idx in range(len(queries_subset)):
#     question = queries_subset.iloc[idx]

#     # Tokenize input question
#     inputs = tokenizer(question, return_tensors="pt", truncation=True, padding=True, max_length=512)

#     # Get model's output
#     with torch.no_grad():
#         outputs = model(**inputs)

#     logits = outputs.logits

#     # Convert logits to probabilities using softmax
#     probabilities = torch.nn.functional.softmax(logits, dim=-1)

#     # Get the predicted class
#     predicted_class = torch.argmax(probabilities, dim=-1).item()
#     question_type.append(predicted_class)

In [ ]:
# df["Question Type"] = question_type
# df

In [ ]:
df

### KNN

In [ ]:
from sklearn.neighbors import KNeighborsRegressor
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score
from sklearn.metrics import mean_squared_error, r2_score

# Separate data into features and target
y = alt_df[['Best Alpha', 'Query ID']]
X = alt_df.drop(['Best Alpha', 'Query ID'], axis=1)

# Split the data into training and test sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

knn_query_ids = y_test

y_train = y_train.drop(["Query ID"], axis=1)
y_test = y_test.drop(["Query ID"], axis=1)

# Scale features using StandardScaler
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# Initialize model
knn = KNeighborsRegressor(n_neighbors=3)

# Fit the model
knn.fit(X_train, y_train)

y_pred = knn.predict(X_test)

mse = mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print(f'Mean Squared Error: {mse}')
print(f'R-squared: {r2}')


In [ ]:
knn_query_ids.iloc[:,1]

In [ ]:
print(y_pred)